## Imports

In [1]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.3 MB/s eta 0:00:00


In [2]:
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from datasets import Dataset as ds
from sklearn.metrics import accuracy_score, classification_report

import textstat
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import os
import pickle

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Linguistic Feature Extraction

In [4]:
def extract_ld_features(text):
    """ extract readability features """
    flesch_reading_ease =  textstat.flesch_reading_ease(text)
    flesch_kincaid_grade =  textstat.flesch_kincaid_grade(text)
    gunning_fog =  textstat.gunning_fog(text)
    smog_index =  textstat.smog_index(text)
    automated_readability_index =  textstat.automated_readability_index(text)

    return [flesch_reading_ease, flesch_kincaid_grade, gunning_fog, smog_index, automated_readability_index]

def get_ld_feature_matrix(texts, cache_file=None):
    if cache_file and os.path.exists(cache_file):
            print(f"Load probabilistic features from {cache_file}")
            return np.load(cache_file)

    feature_matrix = []

    for text in tqdm(texts, desc="extracting ld features"):
        features = extract_ld_features(text)
        feature_matrix.append(features)

    matrix = np.array(feature_matrix)
    if cache_file:
        os.makedirs(os.path.dirname(cache_file), exist_ok=True)
        np.save(cache_file, matrix)
        print(f"Features saved to {cache_file}")

    return matrix

## Token Probabilistic Feature Extraction

In [5]:
FEATS = ["observed", "most_likely", "entropy", "median", "std", "mld", "gini"]

def get_observed(probs, shift_targets, mask, num_valid, eps: float = 1e-14):
    observed = torch.log(
        torch.gather(
            probs, dim=-1, index=shift_targets.unsqueeze(dim=-1)
        ).squeeze(dim=-1)
        + eps
    )
    observed = observed * mask

    # calculate average
    sum_observed = observed.sum(dim=1)

    return (sum_observed / num_valid)

def get_most_likely(probs, mask, num_valid, eps: float = 1e-14):
    most_likely = torch.log(torch.max(probs, dim=-1).values + eps)
    most_likely = most_likely * mask

    sum_most_likely = most_likely.sum(dim=1)

    return (sum_most_likely / num_valid)

def get_entropy(probs, mask, num_valid, eps: float = 1e-14):
    entropy = -torch.sum(probs * torch.log2(probs + eps), dim=-1)
    entropy = entropy * mask

    sum_entropy = entropy.sum(dim=1)

    return (sum_entropy / num_valid)

def get_median(probs, mask, num_valid, eps: float = 1e-14):
    median = torch.log(probs.median(dim=-1).values + eps)
    median = median * mask

    sum_median = median.sum(dim=1)

    return (sum_median / num_valid)

def get_standard_deviation(probs, mask, num_valid, eps: float = 1e-14):
    stdev = torch.log(probs.std(dim=-1) + eps)
    stdev = stdev * mask

    sum_stdev = stdev.sum(dim=1)

    return (sum_stdev / num_valid)

def get_mld(probs, mask, num_valid, eps: float = 1e-14):
    log_mean = torch.log(probs.mean(dim=-1) + eps)
    mean_logs = torch.log(probs + eps).mean(dim=-1)
    mld = log_mean - mean_logs
    mld = mld * mask

    sum_mld = mld.sum(dim=1)

    return (sum_mld / num_valid)

def get_gini(probs, mask, num_valid, eps: float = 1e-14):
    gini = torch.log(1 - probs.square().sum(dim=-1) + eps)
    gini = gini * mask

    sum_gini = gini.sum(dim=1)

    return (sum_gini / num_valid)

class LLMFeatureExtractor:
    def __init__(self, model_name="gpt2"):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)

        self.model.to(self.device)

        self.model.eval()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def extract_tp_features(self, text):
        """
        extracts different token level probabilistic features based on implementation from Team Genaios at sem eval 2024 task 8 
        modified to fit our model architecture 
        """
        encodings = self.tokenizer(text, return_tensors="pt", max_length=self.model.config.max_position_embeddings, truncation=True, padding=True)

        input_ids = encodings.input_ids.to(self.device)
        attention_mask = encodings.attention_mask.to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()

        # labels is the actual token for i+1, so do not look at first token
        shift_labels = input_ids[..., 1:].contiguous()
        shift_attention_mask = attention_mask[..., 1:].contiguous()

        probs = shift_logits.softmax(dim=-1)
        num_valid_tokens = shift_attention_mask.sum(dim=1)
        num_valid_tokens[num_valid_tokens == 0] = 1e-9

        observed = get_observed(probs, shift_labels, shift_attention_mask, num_valid_tokens)

        most_likely = get_most_likely(probs, shift_attention_mask, num_valid_tokens)

        entropy = get_entropy(probs, shift_attention_mask, num_valid_tokens)

        median = get_median(probs, shift_attention_mask, num_valid_tokens)

        standard_deviation = get_standard_deviation(probs, shift_attention_mask, num_valid_tokens)

        mld = get_mld(probs, shift_attention_mask, num_valid_tokens)

        gini = get_gini(probs, shift_attention_mask, num_valid_tokens)

        return [observed.item(), most_likely.item(), entropy.item(), median.item(), standard_deviation.item(), mld.item(), gini.item()]

    def get_tp_feature_matrix(self, texts, cache_file=None):
        """
        get token probabilistic feature representation matrix for all texts in dataset
        """
        if cache_file and os.path.exists(cache_file):
                print(f"Load probabilistic features from {cache_file}")
                return np.load(cache_file)

        feature_matrix = []

        for text in tqdm(texts, desc="extracting tp features"):
            features = self.extract_tp_features(text)
            feature_matrix.append(features)

        matrix = np.array(feature_matrix)
        if cache_file:
            os.makedirs(os.path.dirname(cache_file), exist_ok=True)
            np.save(cache_file, matrix)
            print(f"Features saved to {cache_file}")

        return matrix

## Roberta Embeddings

In [6]:
def get_roberta_embeddings(texts, model, tokenizer, device, cache_file=None):

    if cache_file and os.path.exists(cache_file):
        print(f"Load embeddings from {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    batch_size = 16
    model.eval()
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting RoBERTa embeddings"):
        # Batch von Texten tokenisieren
        batch_texts = texts[i:i+batch_size]
        encoded_inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        batch_inputs = {k: v.to(device) for k, v in encoded_inputs.items()}

        with torch.no_grad():
            outputs = model(**batch_inputs)

            # extract [CLS] Token as representation for Roberta embeddings
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  

            results.extend(cls_embeddings.cpu().numpy())
            
    embeddings = torch.tensor(np.array(results))

    if cache_file:
        os.makedirs(os.path.dirname(cache_file), exist_ok=True)
        with open(cache_file, 'wb') as f:
            pickle.dump(embeddings, f)
        print(f"Embeddings saved to {cache_file}")


    return embeddings

## Dataset to combine Linguistic Features + Roberta Embeddings


In [7]:
class LibertaSet(Dataset):
    """ Linguistic Roberta set combines linguistic features + probabilistic features and roberta embeddings as Dataset for FFN """

    def __init__(self, texts, labels, model, tokenizer, device, normalize=True, rob_cache=None, li_cache=None, pr_cache=None):
        self.texts = texts
        self.labels = labels
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.normalize = normalize

        self.LLM_extractor = LLMFeatureExtractor('gpt2')

        # extract features 

        # token probabilistic 
        self.probabilistic_features = self.LLM_extractor.get_tp_feature_matrix(texts, cache_file=pr_cache)
        
        # linguistic 
        self.linguistic_features = get_ld_feature_matrix(self.texts, cache_file=li_cache)

        # roberta embeddings 
        self.embeddings = get_roberta_embeddings(texts, model, tokenizer, device, cache_file=rob_cache)

        # normalize features 
        if normalize:
            self.linguistic_scaler = StandardScaler()
            self.linguistic_features = torch.tensor(
                self.linguistic_scaler.fit_transform(self.linguistic_features),
                dtype=torch.float32
            )

            self.probabilistic_scaler = StandardScaler()
            self.probabilistic_features = torch.tensor(
                self.probabilistic_scaler.fit_transform(self.probabilistic_features),
                dtype=torch.float32
            )
        else:
            self.linguistic_features = torch.tensor(self.linguistic_features, dtype=torch.float32)
            self.probabilistic_features = torch.tensor(self.probabilistic_features, dtype=torch.float32)


    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        roberta_emb = self.embeddings[idx]
        ling_features = self.linguistic_features[idx]
        prob_features = self.probabilistic_features[idx]

        combined_features = torch.cat([roberta_emb, ling_features, prob_features])

        return combined_features, self.labels[idx]

    def feature_dims(self):
        roberta_dim = self.embeddings.shape[1]
        linguistic_dim = self.linguistic_features.shape[1]
        probabilistic_dim = self.probabilistic_features.shape[1]

        combined_dim = roberta_dim + linguistic_dim + probabilistic_dim

        return {
            'roberta-dim': roberta_dim,
            'linguistic-dim': linguistic_dim,
            'probabilistic-dim': probabilistic_dim,
            'combined-dim': combined_dim,
        }

def get_data(train_path, test_path, model, tokenizer, device):
    train_df = pd.read_json(train_path, lines=True)
    test_df = pd.read_json(test_path, lines=True)

    train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

    train_data = ds.from_pandas(train_df)
    val_data = ds.from_pandas(val_df)

    test_data = ds.from_pandas(test_df)

    # get liberta datasets from data split
    train_texts = train_data["text"]
    train_labels = train_data["label"]

    val_texts = val_data["text"]
    val_labels = val_data["label"]

    test_texts = test_data["text"]
    test_labels = test_data["label"]

    save_dir = '/content/drive/MyDrive/Subtask_A/cache/'

    # get datasets fit to model architecture 
    train_dataset = LibertaSet(train_texts, train_labels, model, tokenizer, device, normalize=True, rob_cache=save_dir+'rob_train.pkl', li_cache=save_dir+'li_train.npy', pr_cache=save_dir+'pr_train.npy')
    val_dataset = LibertaSet(val_texts, val_labels, model, tokenizer, device, normalize=True, rob_cache=save_dir+'rob_val.pkl', li_cache=save_dir+'li_val.npy', pr_cache=save_dir+'pr_val.npy')
    test_dataset = LibertaSet(test_texts, test_labels, model, tokenizer, device, normalize=True, rob_cache=save_dir+'rob_test.pkl', li_cache=save_dir+'li_test.npy', pr_cache=save_dir+'pr_test.npy')

    return train_dataset, val_dataset, test_dataset

## FFN


In [8]:
class CombinedModel(nn.Module):
    """ FFN to process combined inputs of CLS Token and lexdiv features """
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], num_classes=2, dropout_rate=0.3):
        super(CombinedModel, self).__init__()

        self.input_dim = input_dim
        self.num_classes = num_classes

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim

        # output layer
        layers.append(nn.Linear(prev_dim, num_classes))

        self.network = nn.Sequential(*layers)

        self.__init__weights()

    def __init__weights(self):
        """ Use Xavier/Glorot initialization to avoid vanishing/exploding gradients """
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, combined_input):
        logits = self.network(combined_input)

        return logits

## Train and Evaluation Pipeline


In [9]:
def train_model(model, train_loader, val_loader, device, num_epochs, learning_rate=0.001):
    """ Trains the Combined Model """
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    train_losses = []
    val_losses = []
    val_accuracies = []

    best_val_acc = 0.0
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        # training
        model.train()
        train_loss = 0.0

        for batch_features, batch_labels in tqdm(train_loader, desc="training model"):
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            outputs = model(batch_features)

            loss = criterion(outputs, batch_labels)
            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # validation
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch_features, batch_labels in tqdm(val_loader, "validating model"):
                batch_features = batch_features.to(device)
                batch_labels = batch_labels.to(device)

                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                val_loss += loss.item()

                # accuracy
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch_labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(all_labels, all_preds)

        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        scheduler.step(avg_val_loss)

        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_model_state = model.state_dict().copy()
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= 15:
            print("Early stopping triggered. No improvement in validation accuracy for 15 epochs.")
            break

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {avg_train_loss:.4f}, '
              f'Val Loss: {avg_val_loss:.4f}, '
              f'Val Acc: {val_accuracy:.4f}')

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f'Best validation accuracy: {best_val_acc:.4f}')

    return model, train_losses, val_losses, val_accuracies

def evaluate_model(model, test_loader, device):
    """ evaluates CombinedModel on test data """
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_features, batch_labels in tqdm(test_loader, desc="evaluating on testing set"):
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())

    test_accuracy = accuracy_score(all_labels, all_preds)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds))

    return test_accuracy, all_preds, all_labels

## Run training and testing

In [10]:
def run_complete():
    train_path = '/content/drive/MyDrive/Subtask_A/subtaskA_train_monolingual.jsonl'
    test_path = '/content/drive/MyDrive/Subtask_A/subtaskA_dev_monolingual.jsonl'

    model_path = '/content/drive/MyDrive/Subtask_A/checkpoints/best_roberta'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    roberta_model = AutoModel.from_pretrained(model_path)
    roberta_model.to(device)

    print(f"Model device: {next(roberta_model.parameters()).device}")

    train_data, val_data, test_data = get_data(train_path, test_path, roberta_model, tokenizer, device)

    input_dim = train_data.feature_dims()['combined-dim']
    rob_dim = train_data.feature_dims()['roberta-dim']
    ling_dim = train_data.feature_dims()['linguistic-dim']
    prob_dim = train_data.feature_dims()['probabilistic-dim']
    print(f"Roberta Dimension: {rob_dim}")
    print(f"Linguistic Dimension: {ling_dim}")
    print(f"Probabilistic Dimension: {prob_dim}")
    print(f"Input Dimension: {input_dim}")

    batch_size = 32
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    model = CombinedModel(
        input_dim=input_dim,
        hidden_dims=[512, 256, 128],
        num_classes=2,
        dropout_rate=0.3
    )

    print(model)

    print("Starting training...")
    trained_model, train_losses, val_losses, val_accuracies = train_model(
        model, train_loader, val_loader, device, num_epochs=50, learning_rate=0.001
    )

    print("Start testing...")
    test_accuracy, predictions, true_labels = evaluate_model(trained_model, test_loader, device)

    torch.save(trained_model.state_dict(), 'combined_model-3.pth')
    print("Model saved as 'combined_model-3.pth'")
    print("Probabilistic + Linguistic + Roberta Model")

## Test combined model on test data


In [13]:
def test():
  test_path = '/content/drive/MyDrive/Subtask_A/subtaskA_monolingual.jsonl'
  test_df = pd.read_json(test_path, lines=True)
  test_data = ds.from_pandas(test_df)

  test_texts = test_data["text"]
  test_labels = test_data["label"]

  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  print(f"Using device: {device}")

  rob_model_path = '/content/drive/MyDrive/Subtask_A/checkpoints/best_roberta'
  tokenizer = AutoTokenizer.from_pretrained(rob_model_path)
  roberta_model = AutoModel.from_pretrained(rob_model_path)
  roberta_model.to(device)

  save_dir = '/content/drive/MyDrive/Subtask_A/cache/'
  test_dataset = LibertaSet(test_texts, test_labels, roberta_model, tokenizer, device, normalize=True, rob_cache=save_dir+'rob_testset.pkl', li_cache=save_dir+'li_testset.npy', pr_cache=save_dir+'pr_testset.npy')
  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

  input_dim = test_dataset.feature_dims()['combined-dim']

  combined_model = CombinedModel(
    input_dim=input_dim,
    hidden_dims=[512, 256, 128],
    num_classes=2,
    dropout_rate=0.3
  )

  combined_model_path = '/content/combined_model-3.pth'
  checkpoint = torch.load(combined_model_path, map_location=device)

  if 'model_state_dict' in checkpoint:
    print("state dict")
    combined_model.load_state_dict(checkpoint['model_state_dict'])
  else:
    combined_model.load_state_dict(checkpoint)

  combined_model.to(device)

  print(f"Model device: {next(combined_model.parameters()).device}")

  evaluate_model(combined_model, test_loader, device)



In [11]:
run_complete()

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading adapter weights from /content/drive/MyDrive/Subtask_A/checkpoints/best_roberta led to unexpected keys not found in the model: classifier.dense.bias, classifier.dense.weight, classifier.out_proj.bias, classifier.out_proj.weight, roberta.encoder.layer.0.attention.self.query.lora_A.default.weight, roberta.encoder.layer.0.attention.self.query.lora_B.default.weight, roberta.encoder.layer.0.attention.self.value.lora_A.default.weight, roberta.encoder.layer.0.attention.self.value.lora_B.default.weight, roberta.encoder.layer.1.attention.self.query.lora_A.default.weight, roberta.encoder.layer.1.attention.self.query.lora_B.default.weight, roberta.encoder.layer.1.attention.self.value.lora_A.default.weight, roberta.encod

...

Epoch [17/50], Train Loss: 0.0223, Val Loss: 0.0239, Val Acc: 0.9916


validating model: 100%|██████████| 749/749 [00:00<00:00, 749.73it/s]


Early stopping triggered. No improvement in validation accuracy for 15 epochs.
Best validation accuracy: 0.9939
Start testing...


evaluating on testing set: 100%|██████████| 157/157 [00:00<00:00, 782.88it/s]


Test Accuracy: 0.7564
Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.93      0.79      2500
           1       0.89      0.58      0.71      2500

    accuracy                           0.76      5000
   macro avg       0.79      0.76      0.75      5000
weighted avg       0.79      0.76      0.75      5000

Model saved as 'combined_model-3.pth'
Probabilistic + Linguistic + Roberta Model


In [14]:
test()

Using device: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading adapter weights from /content/drive/MyDrive/Subtask_A/checkpoints/best_roberta led to unexpected keys not found in the model: classifier.dense.bias, classifier.dense.weight, classifier.out_proj.bias, classifier.out_proj.weight, roberta.encoder.layer.0.attention.self.query.lora_A.default.weight, roberta.encoder.layer.0.attention.self.query.lora_B.default.weight, roberta.encoder.layer.0.attention.self.value.lora_A.default.weight, roberta.encoder.layer.0.attention.self.value.lora_B.default.weight, roberta.encoder.layer.1.attention.self.query.lora_A.default.weight, roberta.encoder.layer.1.attention.self.query.lora_B.default.weight, roberta.encoder.layer.1.attention.self.value.lora_A.default.weight, roberta.encod

Using device: cuda


extracting tp features: 100%|██████████| 34272/34272 [10:17<00:00, 55.50it/s]


Features saved to /content/drive/MyDrive/Subtask_A/cache/pr_testset.npy


...

Features saved to /content/drive/MyDrive/Subtask_A/cache/li_testset.npy


Extracting RoBERTa embeddings: 100%|██████████| 2142/2142 [03:56<00:00,  9.06it/s]


Embeddings saved to /content/drive/MyDrive/Subtask_A/cache/rob_testset.pkl
Model device: cuda:0


evaluating on testing set: 100%|██████████| 1071/1071 [00:01<00:00, 760.19it/s]


Test Accuracy: 0.8864
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.93      0.89     16272
           1       0.93      0.85      0.89     18000

    accuracy                           0.89     34272
   macro avg       0.89      0.89      0.89     34272
weighted avg       0.89      0.89      0.89     34272

